In [14]:
import duckdb
import os

In [15]:
FACT = "../../data_lake/gold/facts/fact_orders.parquet"
CUSTOMERS = "../../data_lake/gold/dimensions/dim_customers.parquet"
SELLERS = "../../data_lake/gold/dimensions/dim_sellers.parquet"
PRODUCTS = "../../data_lake/gold/dimensions/dim_products.parquet"

In [16]:
GEOLOCATION = "../../data_lake/silver/geolocation/geolocation_silver.parquet"

In [17]:
OUTPUT_DIR = "../../data_lake/ml/datasets"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT = f"{OUTPUT_DIR}/raw_ml_dataset.parquet"

con = duckdb.connect()

In [18]:
OUTPUT_DIR = "../../data_lake/ml/datasets"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_NEW = f"{OUTPUT_DIR}/raw_ml_dataset_new_version.parquet"

In [38]:
con.execute(f"""
COPY (
SELECT
    f.order_id,
    f.is_late_delivery,

    f.price,
    f.freight_value,
    f.total_item_value,

    p.product_weight_g,
    p.product_volume_cm3,
    p.product_category_name,

    s.seller_state,
    c.customer_state,
            
    s.seller_city,
    c.customer_city,
            
    s.zip_code_prefix as seller_zip_code_prefix,
    c.zip_code_prefix as customer_zip_code_prefix,

    f.purchase_ts

FROM read_parquet('{FACT}') f

LEFT JOIN read_parquet('{PRODUCTS}') p
ON f.product_id = p.product_id

LEFT JOIN read_parquet('{SELLERS}') s
ON f.seller_id = s.seller_id

LEFT JOIN read_parquet('{CUSTOMERS}') c
ON f.customer_id = c.customer_id

WHERE f.is_delivered = 1
)
TO '{OUTPUT}' (FORMAT PARQUET)
""")

In [43]:
import pandas as pd
df = pd.read_parquet(OUTPUT)
df.head()

,order_id,is_late_delivery,price,freight_value,total_item_value,product_weight_g,product_volume_cm3,product_category_name,seller_state,customer_state,seller_city,customer_city,seller_zip_code_prefix,customer_zip_code_prefix,purchase_ts
0,3a6d41a4fe7a8e841b7c8e4b77d4e301,0,53.99,15.13,69.12,200.0,3328.0,perfumaria,SP,ES,santo andre,cachoeiro de itapemirim,09015,29311,2017-11-24 11:29:52
1,f07b0af882cdeef616def36fd78fbb23,0,239.90,16.43,256.33,335.0,2992.0,relogios_presentes,SP,RJ,sumare,paraiba do sul,13170,25850,2017-07-16 19:20:47
2,26691e37091554c74df671ce610ead4c,0,116.90,16.58,133.48,1750.0,4500.0,cama_mesa_banho,SP,MG,ibitinga,uberlandia,14940,38400,2017-11-24 21:07:00
3,75559e14f308f4ff08427c567ae0a370,1,138.00,13.99,151.99,1200.0,11907.0,esporte_lazer,SP,SP,guara,santos,14580,11045,2018-02-12 13:53:31
4,5fbb36c2e55d34d7799cb74e65d73a62,0,72.00,16.69,88.69,1800.0,20900.0,utilidades_domesticas,SP,MG,sao paulo,belo horizonte,09560,30431,2018-07-03 09:09:36


In [39]:
df.count()

order_id                    110197
is_late_delivery            110197
price                       110197
freight_value               110197
total_item_value            110197
product_weight_g            110179
product_volume_cm3          110179
product_category_name       108660
seller_state                110197
customer_state              110197
seller_zip_code_prefix      110197
customer_zip_code_prefix    110197
purchase_ts                 110197
dtype: int64

In [41]:
con.execute(f"""
COPY (
WITH seller_geolocation AS (
SELECT
    f.*,
            
    g.avg_lat as seller_avg_lat,
    g.avg_lng as seller_avg_lng,

    f.purchase_ts

FROM read_parquet('{OUTPUT}') f

LEFT JOIN read_parquet('{GEOLOCATION}') g
ON f.seller_zip_code_prefix = g.zipp_code_prefix
AND f.seller_state = g.geolocation_state
AND f.seller_city = g.city
)
SELECT
    sg.*,
            
    g.avg_lat as customer_avg_lat,
    g.avg_lng as customer_avg_lng

FROM seller_geolocation sg

LEFT JOIN read_parquet('{GEOLOCATION}') g
ON sg.customer_zip_code_prefix = g.zipp_code_prefix
AND sg.customer_state = g.geolocation_state
AND sg.customer_city = g.city

)
TO '{OUTPUT_NEW}' (FORMAT PARQUET)
""")

In [42]:
import pandas as pd
df_2 = pd.read_parquet(OUTPUT_NEW)
df_2.head()
df_2.count()

order_id                    110197
is_late_delivery            110197
price                       110197
freight_value               110197
total_item_value            110197
product_weight_g            110179
product_volume_cm3          110179
product_category_name       108660
seller_state                110197
customer_state              110197
seller_city                 110197
customer_city               110197
seller_zip_code_prefix      110197
customer_zip_code_prefix    110197
purchase_ts                 110197
seller_avg_lat              107185
seller_avg_lng              107185
purchase_ts_1               110197
customer_avg_lat            109865
customer_avg_lng            109865
dtype: int64